# Train CatBoost classifier

Input file (place in the same folder):
- `ml_train_dataset.xlsx`

Outputs (used by `2_abm_demo.ipynb` and `3_scenarios_and_sensitivity.ipynb`):
- `final_catboost_model.cbm`
- `final_catboost_model.meta.json`

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool
import optuna
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import json

features = [
    'FBO type',
    'Product type',
    'Operation type',
    'Former Inspection result',
    'FBO Size',
    'Inspection interval (days)'
]
cat_features = [
    'FBO type',
    'Product type',
    'Operation type',
    'Former Inspection result',
    'FBO Size'
]

## 1. Load data

In [ ]:
df = pd.read_excel('ml_train_dataset.xlsx')

# Behaviour change (-3 to 3) grouped into three classes:
# 0 = improvement, 1 = no change, 2 = deterioration
y = df['Behavior Change'].map({-3: 0, -2: 0, -1: 0, 0: 1, 1: 2, 2: 2, 3: 2})

X = df[features]

print(y.value_counts().sort_index())

## 2. Split data

In [ ]:
#  60% training, 20% validation, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features)

## 3. Class weights

In [ ]:
# Inverse-frequency weights
classes = sorted(y.unique())
train_counts = y_train.value_counts()
total = len(y_train)
class_weights = []
for c in classes:
    class_weights.append(total / (len(classes) * train_counts[c]))
print("class weights:", class_weights)

## 4. Hyperparameter tuning 

In [ ]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_seed': 42,
        'loss_function': 'MultiClass',
        'eval_metric': 'MultiClass',
        'class_weights': class_weights,
        'early_stopping_rounds': 50,
        'verbose': False,
    }
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    # validation macro-F1 is the score Optuna tries to maximise
    preds = model.predict(val_pool).astype(int).ravel()
    return f1_score(y_val, preds, average='macro')

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print("Best params:", study.best_trial.params)

## 5. Final model 

In [ ]:
best_params = study.best_trial.params
best_params.update({
    'loss_function': 'MultiClass',
    'class_weights': class_weights,
    'random_seed': 42,
    'early_stopping_rounds': 50,
    'verbose': 100,
})
final_model = CatBoostClassifier(**best_params)
final_model.fit(train_pool, eval_set=val_pool, use_best_model=True)

# Performance 
val_preds = final_model.predict(val_pool).astype(int).ravel()
print("Validation report:\n", classification_report(y_val, val_preds, zero_division=0))
print("Validation confusion matrix:\n", confusion_matrix(y_val, val_preds))

test_preds = final_model.predict(test_pool).astype(int).ravel()
print("Test report:\n", classification_report(y_test, test_preds, zero_division=0))
print("Test confusion matrix:\n", confusion_matrix(y_test, test_preds))
print("Test macro-F1:", f1_score(y_test, test_preds, average='macro'))
print("Test accuracy:", accuracy_score(y_test, test_preds))

## 6. Save model for use in ABM

In [ ]:
final_model.save_model('final_catboost_model.cbm')

# The ABM reads -1 as improvement, 0 as no change and +1 as deterioration,
# so the model classes (0/1/2) are mapped to these values in the meta file
meta = {
    'feature_order': features,
    'categorical_features': cat_features,
    'target_mapping': {0: -1, 1: 0, 2: 1},
}
with open('final_catboost_model.meta.json', 'w') as f:
    json.dump(meta, f, indent=2)